# TrkA_88 재현 실패 원인 비교 · 진단 v0.1.0

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_protein/blob/main/diagnostics/90_af2_conditions_T4.ipynb?v=0.1.0)

T4에서 전체 실행하면 원본 dl_binder_design AF2 코드로 아래 11개 조건을 순서대로 비교합니다. 같은 세션의 설치와 가중치를 재사용합니다. 각 조건은 별도 프로세스에서 실행하며, 실패해도 다음 조건을 진행합니다. AF2를 11회 실행하므로 수십 분이 걸릴 수 있습니다.

| 조건 | 원본 서열 입히기·시드 0·recycle 3 기준에서 바꾸는 항목 |
| --- | --- |
| mutate_baseline | 기존 노트북의 mutate_residue 방식으로 서열 준비 |
| original_thread | 원본 ProteinMPNN 코드의 thread_mpnn_seq 함수를 사용 |
| seed_1 | 난수 시드 1 |
| seed_2 | 난수 시드 2 |
| recycles_12 | recycle 12회 |
| no_initial_guess | 설계 좌표로 초기화하는 기능 해제 |
| empty_target_template | 표적 템플릿의 원자 마스크와 좌표를 0으로 설정 |
| published_coordinates_control | 공개 AF2 복합체 좌표를 시작 입력으로 제공 |
| published_target_control | 공개 AF2의 표적 원자 좌표만 원래 설계 좌표계에 맞춰 제공 |
| highest_precision | JAX 행렬 곱 정밀도를 highest로 설정 |
| original_thread_repeat | 원본 조건을 같은 시드로 다시 실행 |

공개 AF2 좌표를 사용하는 두 조건은 참고 구조를 제공하는 대조 실험입니다. 서열에서 시작하는 조건과 입력이 다르다는 점을 표에 함께 기록합니다.

모든 결과는 원래 설계 구조를 기준으로 TrkA를 고정해 결합체의 위치·방향 RMSD를 계산합니다. 공개 AF2 구조와의 차이, 결합면 PAE, 결합체 pLDDT도 표시합니다. 논문 보고값은 약 1.4 Å입니다.

ZIP에는 조건별 입력 PDB, 실제 원시·전처리 특징 배열, 초기 좌표, 모델 설정, AF2 신뢰도 배열, 예측 PDB, 로그, 실패 기록이 들어갑니다. feature_differences.json은 original_thread와 각 조건의 입력 배열을 비교합니다. 가중치와 원 코드의 SHA-256도 저장합니다.

[원본 AF2 코드](https://github.com/nrbennet/dl_binder_design/blob/cafa3853ac94dceb1b908c8d9e6954d71749871a/af2_initial_guess/predict.py) · [공개 서열·구조](https://doi.org/10.6084/m9.figshare.22662094.v1)


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import subprocess
import sys

checkout = Path(mkdtemp(prefix="trka88_main_", dir="/content")) / "repo"
clone = subprocess.run(
    ["git", "clone", "--depth", "1", "--single-branch", "--branch", "main",
     "https://github.com/HisameOgasahara/tmp_protein.git", str(checkout)],
    capture_output=True, text=True,
)
if clone.returncode:
    print(clone.stderr)
    clone.check_returncode()
revision = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
print(f"main: {revision[:12]}")


In [ ]:
import json
import pandas as pd
from IPython.display import display

log = checkout.parent / "trka88_diagnostics.log"
result_zip = None
with log.open("w", encoding="utf-8") as handle:
    with subprocess.Popen(
        [sys.executable, "-u", str(checkout / "src/af2_filter/diagnostic_workflow.py"), revision],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
    ) as process:
        for line in process.stdout:
            handle.write(line)
            handle.flush()
            print(line, end="", flush=True)
            if line.strip().endswith(".zip"):
                result_zip = Path(line.strip())
        if process.wait():
            raise RuntimeError(f"Diagnostics failed. Full log: {log}")
if result_zip is None or not result_zip.is_file():
    raise FileNotFoundError(f"Missing diagnostic ZIP. Full log: {log}")
summary = result_zip.with_suffix("") / "diagnostic_summary.json"
if summary.is_file():
    rows = json.loads(summary.read_text())
    columns = [
        "id", "status", "rmsd_vs_original_design", "rmsd_vs_published_af2",
        "pae_interaction", "plddt_binder", "seconds",
    ]
    table = pd.DataFrame(rows).reindex(columns=columns)
    display(table.round(3))
else:
    print("설치 단계 오류 기록이 ZIP에 저장됐습니다.")
print(result_zip)


In [ ]:
from google.colab import files
files.download(str(result_zip))
